# Data Visualisation et Analyse de Cohortes sur Olist.

### **_Un peu de contexte..._**

**La plateforme e-commerce Olist connecte les propriétaires de petits commerces brésiliens (les vendeurs) avec des consommateurs partout au Brésil.** Comprendre ce qui pousse les vendeurs à continuer d'utiliser la plateforme une fois qu'ils ont effectué leur première vente est une **question essentielle** pour Olist !

**L'analyse de cohortes vise à regrouper les individus en 'cohortes' selon une caractéristique commune, puis à étudier l'évolution de ces cohortes au fil du temps.**

Un petit exemple pour comprendre cela :

> _Il est possible d'étudier l'impact d'une **campagne marketing** sur différentes cohortes, d'analyser le taux d'attrition des cohortes, etc._

### **_La Mission..._**

**Olist vous a engagé comme Data Analyst pour réaliser une analyse de cohortes sur leurs données 2017-2018.**

Ici, les clients d'Olist sont les vendeurs du site. Il s'agit de les répartir en cohortes correspondant au mois de leur première vente. Il ne s'agit pas tout à fait d'une analyse de cohortes classique : ici, l'objectif sera de suivre la **proportion de vendeurs de chaque cohorte qui continue à utiliser la plateforme** dans les mois suivants leur première vente.

### **_Les données à analyser..._**

Les données à utiliser sont disponibles sur la plateforme ainsi que sur [Kaggle](https://www.kaggle.com/olistbr/brazilian-ecommerce). Nous utiliserons les tables `olist_sellers_dataset`, `olist_orders_dataset` et `olist_order_items_dataset`.

> _N'hésitez pas à consulter la documentation sur les tables disponible sur Kaggle : [sur ce lien](https://www.kaggle.com/olistbr/brazilian-ecommerce)._


### On commence par importer les packages nécéssaires.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

###  Fusionner les tables à analyser.

Pour cette analyse, nous n'aurons besoin que de **deux variables :**

- Les **IDs** des vendeurs dans `olist_sellers_dataset`.
- La **date** de chaque vente (`order_purchase_timestamp`) dans `olist_orders_dataset`.

**Fusionner les tables et ne conserver que les 2 colonnes pertinentes.**

Pour connecter les deux tables, il est indispensable d'utiliser `olist_order_items_dataset` comme intermédiaire (avec `order_id`).

#### Les tables sont importées via un repository GitHub, exécuter la cellule ci-dessous.

In [ ]:
df_orders = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_orders_dataset.csv')
df_order_items = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_items_dataset.csv')
df_sellers = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_sellers_dataset.csv')

**Maintenant que les tables sont importées dans le notebook, il va falloir joindre les données.**

In [ ]:
# Nous pouvons joindre la donnée avec la méthode '.merge()' de Pandas et ce, de deux manières :

# 1ère Méthode : (pd.merge()).

df_intermediaire = pd.merge(df_orders, df_order_items, on='order_id') # Nous renseignons dans la fonction les tables, ainsi que la clé de jointure.
df = pd.merge(df_intermediaire, df_sellers, on='seller_id')

df = df[['seller_id', 'order_purchase_timestamp']] # Nous sélectionnons les colonnes qui nous intéressent.
display(df)

# 2ème Méthode : (df.merge()).

df = df_orders.merge(df_order_items, on='order_id').merge(df_sellers, on='seller_id') # Nous pouvons compiler les méthodes (.merge()).

df = df[['seller_id', 'order_purchase_timestamp']] # Nous sélectionnons les colonnes qui nous intéressent.
display(df)

###  Filtrer les dates.


**Convertir la colonne de dates au bon type, et conserver uniquement les données de Jan 2017 à Sept 2018.**

Il est possible (mais optionnel) d'utiliser les méthodes `df.assign()` et `df.query()`. La méthode `df.assign()` permet de créer une nouvelle colonne à partir d'une fonction lambda. `df.query()` permet de filtrer un dataframe selon la date.

**Ces méthodes sont bonus, elles permettent d'être plus concis mais ne sont pas du tout obligatoires.**

In [ ]:
# Cellule de conversion en type date, de manière sont spécifier :

# 1ère Méthode : La méthode classique.

df['date'] = pd.to_datetime(df['order_purchase_timestamp']) # Création d'une colonne 'date' du type datetime.

# Nous pouvons maintenant supprimer la colonne 'order_purchase_timestamp'.
df = df.drop('order_purchase_timestamp', axis=1)

df = df[ (df['date'] >= '2017') & (df['date'] <= '09-2018') ] # Conditions de filtrage qui permettent de filtrer le dataframe de Jan 2017 à Sept 2018.
display(df)

# 2ème Méthode : La méthode élégante.

df = (
    df
    .assign(date = lambda x: pd.to_datetime(x['date']))  # Convertit la colonne 'date' en type datetime pour s'assurer que les dates sont correctement formatées et manipulables pour les opérations suivantes.
    .query('date >= "2017-01-01" & date < "2018-09-01"')  # Filtre les enregistrements pour ne garder que ceux dont la date est comprise entre le 1er janvier 2017 et le 1er septembre 2018.
)

display(df)

### Constitution des cohortes.


#### **Attention, les cohortes doivent être définies selon le mois de la première vente, c'est-à-dire, une cohorte par mois.**

Pour créer les cohortes, commencer par **extraire** le mois de vente à partir des **dates complètes.**

Ensuite, ordonner les données selon le **mois de vente** et **supprimer les doublons** dans la colonne `seller_id`, afin de ne conserver que la première apparition de chaque vendeur...

> _Pour extraire la période annee.mois, on peut utiliser la méthode `df['col'].dt.to_period('M')`._

In [ ]:
# Encore une fois, on vous introduit deux méthodes différentes :

# 1ère Méthode : La classique.

cohort_vendeur = df.copy() # Nous stockons dans la variable 'seller_cohort' une copie des données source.

# Pour s'assurer que la première vente de chaque vendeur est en premier...
cohort_vendeur = cohort_vendeur.sort_values(by='date') # Trions les données par notre nouvelle colonne 'date'.

# Pour garder uniquement la première occurrence, la première vente de chaque vendeur...
cohort_vendeur = cohort_vendeur.drop_duplicates(subset='seller_id', keep='first') # Supprimons les doublons avec '.drop_duplicates()'.

# Pour extraire le mois de la première vente...
cohort_vendeur['cohort'] = cohort_vendeur['date'].dt.to_period('M') # Nous créeons une nouvelle colonne 'cohort' référençant le mois au format YYYY-MM.

cohort_vendeur = cohort_vendeur.reset_index(drop=True) # Réinitialisation des index après les manipulations précédentes.
cohort_vendeur = cohort_vendeur.drop('date', axis=1) # Suppression de la colonne 'date', car remplacée par la colonne 'cohort'.

display(cohort_vendeur)

# 2ème Méthode : La compliquée.

cohort_vendeur = (
    df
    .assign(cohort = lambda x: x['date'].dt.to_period('M'))  # Crée une nouvelle colonne 'cohort' en extrayant le mois et l'année de la colonne 'date' pour chaque enregistrement.
    .sort_values('date')  # Trie les enregistrements par date, du plus ancien au plus récent, pour identifier la première vente de chaque vendeur.
    .drop_duplicates('seller_id')  # Supprime les doublons basés sur 'seller_id', ne gardant que la première occurrence (première vente) pour chaque vendeur.
    .reset_index()  # Réinitialise l'index pour obtenir une série d'index consécutifs après le tri et la suppression des doublons.
    .loc[:, ['seller_id', 'cohort']]  # Sélectionne uniquement les colonnes 'seller_id' et 'cohort' pour le DataFrame final, filtrant les autres colonnes.
)

display(cohort_vendeur)

### Visualiser le nombre de vendeurs dans chaque cohorte grâce à un diagramme en bâtons.

Avant de continuer l'analyse, nous allons **visualiser nos données**. En particulier, nous aimerions avoir une idée de la taille de chaque cohorte.

**Nous allons compter le nombre de vendeurs uniques dans chaque cohorte, puis utiliser seaborn pour créer un bar chart.**

_Souvenez-vous de la fonction `sns.catplot()` !_

In [ ]:
taille_cohorts = (cohort_vendeur
                .groupby('cohort')['seller_id']
                .nunique()
                .reset_index()
                )
display(taille_cohorts)

In [ ]:
# Il est également possible de résumer le code de manière plus simple.

taille_cohorts = cohort_vendeur['cohort'].value_counts(sort=False).reset_index() # On définit la taille des cohortes avec la méthode '.value_counts()'.
taille_cohorts.columns = ['cohort', 'seller_id'] # On ressort seulement les colonnes qui nous intéressent.

display(taille_cohorts)

In [ ]:
# Visualisons les résultats.

sns.catplot(data=taille_cohorts, x='cohort', y='seller_id', kind='bar', aspect=2) # On génère un categorical plot.
plt.xticks(rotation=45) # Rotation des éléments dans l'axe des abcisses.

In [ ]:
# On peut également visualiser la taille des cohortes avec une palette de couleurs dynamique.

# Générer une palette de couleurs dégradée...
colors = sns.color_palette("Greens_d", len(taille_cohorts))  # Création d'une couleur différente pour chaque cohorte.

# Classer les cohortes en fonction du nombre de vendeurs uniques...
rank = taille_cohorts['seller_id'].argsort().argsort()  # Obtenir le rang de chaque cohorte en fonction du nombre de vendeurs uniques.

# Créer un graphique en barres pour visualiser la taille des cohortes...
sns.catplot(data=taille_cohorts, y='seller_id', x='cohort',
            kind='bar', aspect=2, palette=np.array(colors)[rank])  # Tracer un graphique en barres avec une couleur pour chaque cohorte selon son rang.

plt.xticks(rotation=45) # Rotation des éléments dans l'axe des abscisses.
plt.show()

On observe une **certaine variabilité** dans le nombre de nouveaux vendeurs par mois, mais aucun **effet saisonnier** ne semble évident.

#### **Étudions l'attrition de ces cohortes.**

### Pour chaque vendeur :

- Calculer **la différence** entre chacune des **ventes** et sa **cohorte** (mois de première vente).

- **Fusionner la table** contenant les **noms** de cohortes (question précédente) avec la **table** initiale (contenant toutes les ventes).

> _Il est maintenant **possible de calculer** la différence (en mois) entre la date de chaque vente et la cohorte du vendeur (mois de première vente)._

#### **Vous pourrez appeler cette différence `delta`.**

In [ ]:
# Détaillons encore deux méthodes :

# 1ère Méthode : La classique.


cohort_df = pd.merge(df, cohort_vendeur, on='seller_id') # Jointure entre les deux dataframes qui nous intéressent.

cohort_df['date'] = cohort_df['date'].dt.to_period('M') # On re-crée la colonne 'date'.

# Puis nous créons la colonne delta en soustrayant les dates formatées en valeur entière avec les cohortes également formatées en valeur entière.

cohort_df['delta'] = cohort_df['date'].astype(int) - cohort_df['cohort'].astype(int)

display(cohort_df)

# 2ème Méthode : la méthode élégante, (mais pas forcément plus claire à la lecture !).

# Création du DataFrame des cohortes en combinant les informations de vente et de cohortes des vendeurs :

cohort_df = (
    # Fusion des DataFrames 'df' et 'seller_cohort' sur la colonne 'seller_id' (l'identifiant des vendeurs).
    pd.merge(df, cohort_vendeur, on='seller_id')

    # Création de nouvelles colonnes via la méthode assign :
    # 'date_of_sale' est la date de la vente convertie au format de période mensuelle (Year-Month).
    .assign(date_of_sale = lambda x: x.date.dt.to_period('M'),

            # Calcul du délai (delta) entre la date de vente et la cohorte d'entrée du vendeur.
            # On soustrait la cohorte (mois d'entrée) à la date de vente, puis on extrait la valeur numérique.
            delta = lambda x: (x.date_of_sale - x.cohort).apply(lambda x: x.n))

    # Sélection des colonnes pertinentes pour l'analyse : 'seller_id', 'cohort', 'date_of_sale', et 'delta'.
    .loc[:, ['seller_id', 'cohort', 'date_of_sale', 'delta']]
)

display(cohort_df)

###  Étude de l'attrition des vendeurs, étude du Churn.

- Pour chaque cohorte, **compter le nombre de vendeurs uniques par delta.**

- **Convertir ce nombre** pour l'exprimer en **pourcentage** de la taille de la cohorte.

- Ces calculs nous **permettent** d'observer quelle proportion de chaque cohorte continue de **réaliser des ventes** au fil du temps.

> _Il faudra d'abord **grouper votre table par cohorte** et par delta pour obtenir le nombre de vendeurs uniques par cohorte et par delta. Ensuite, **on pourra fusionner cette table** avec les totaux calculés dans la question précédente, afin de calculer les pourcentages par rapport au total de chaque cohorte._

In [ ]:
# Deux méthodes :

# 1ère Méthode : La classique.

# Grouper les données par cohorte et delta (le nombre de mois après la première vente), et compter le nombre de vendeurs uniques pour chaque combinaison.
cohort_df_delta = cohort_df.groupby(['cohort', 'delta'])['seller_id'].nunique()

# Réinitialiser l'index pour transformer le résultat en DataFrame plat.
cohort_df_delta = cohort_df_delta.reset_index()

# Fusionner le DataFrame résultant avec 'size_cohorts' pour obtenir la taille totale de chaque cohorte.
cohort_df_delta = cohort_df_delta.merge(taille_cohorts, on='cohort')

# Renommer les colonnes pour une meilleure compréhension.
cohort_df_delta.columns = ['cohort', 'delta', 'per_delta', 'total']

# Calculer la proportion de vendeurs restants par rapport au total de la cohorte.
cohort_df_delta['proportion'] = (cohort_df_delta['per_delta'] / cohort_df_delta['total'])*100

# Supprimer les colonnes intermédiaires ('per_delta' et 'total') qui ne sont plus nécessaires.
cohort_df_delta = cohort_df_delta.drop(columns=['per_delta', 'total'])

display(cohort_df_delta)

# 2ème Méthode : L'élégante.

# Grouper les données par cohorte et delta, puis compter le nombre de vendeurs uniques pour chaque combinaison.
cohort_df_delta = (cohort_df
                 .groupby(['cohort', 'delta'])[['seller_id']]  # Groupement par 'cohort' et 'delta'
                 .nunique()  # Compter le nombre de vendeurs uniques dans chaque groupe

                 # Ajout d'une nouvelle colonne 'total', qui contient la taille totale de la cohorte
                 .assign(total = lambda x: x.groupby('cohort')['seller_id']
                                            .transform(lambda x: x[0]),  # Pour chaque cohorte, le premier vendeur est la taille totale de la cohorte

                         # Calcul de la proportion : nombre de vendeurs / taille totale * 100
                         proportion = lambda x: 100 * x.seller_id / x.total)

                 # Réinitialiser l'index pour avoir un DataFrame classique
                 .reset_index()

                 # Sélectionner uniquement les colonnes pertinentes : 'cohort', 'delta', et 'proportion'
                 .loc[:, ['cohort', 'delta', 'proportion']]
                )

display(cohort_df_delta)

### Pivoter la table.

Utiliser  `df.pivot()` (ou `df.pivot_table`) pour **obtenir une table avec les valeurs de delta** en colonnes, les différentes cohortes en lignes et les proportions calculées comme valeurs.


In [ ]:
# Transformer les données en une table pivotée :

cohort_pivot = cohort_df_delta.pivot(columns='delta',   # Chaque colonne représentera un delta (mois écoulé depuis la première vente).
                                     index='cohort',    # Chaque ligne correspondra à une cohorte (mois de la première vente).
                                     values='proportion')  # Les valeurs dans la table seront la proportion de vendeurs restants.
cohort_pivot

### Visualiser les cohortes finales et les taux d'attrition grâce à une heatmap.

> _Vous pourrez utiliser `plt.figure(figsize=(12, 8))` en première ligne de votre cellule pour ajuster la taille du graphique. Vous pourrez **également** ajouter dans la fonction `sns.heatmap()` les arguments : `cmap='crest', annot=True, fmt='.0f'`, pour un résultat **plus esthétique !**_

In [ ]:
# Définir la taille de la figure (12 de largeur et 8 de hauteur) :

plt.figure(figsize=(12, 8))

sns.heatmap(cohort_pivot,    # Utiliser les données pivotées pour la heatmap.
            cmap='crest',    # Utiliser la palette de couleurs 'crest' pour colorer la heatmap.
            annot=True,      # Annoter chaque cellule avec sa valeur numérique.
            fmt='.0f')       # Formater les annotations en entier, sans décimale.

plt.title("Olist Cohort Analysis", fontweight='bold')  # Ajouter un titre en gras à la heatmap.

### Que déduire de cette visualisation ?

Il est **intéressant de noter** que la **1ère** et **7ème** cohortes ont été relativement plus **fidèles à Olist** que les autres.

**Ce sont les seules avec un taux de participation 6 mois après la première vente strictement supérieur à 50%.**

> **Il appartient désormais aux équipes d'Olist (Marketing, Sales) de comprendre pourquoi ces cohortes ont un taux d'attrition nettement plus faible que les autres.**

# Bonus : Différents types de segmentation.

### Commençons par un pré-traitement des données.

Commencez par **exécuter les cellules** ci-dessous pour charger les données, **fusionner les tables** et effectuer toutes les étapes de pré-traitement.

> _Ci-dessous, les données sont lues directement depuis GitHub pour éviter le téléchargement. La fonction `clean_olist` permet de réaliser le pré-traitement en une seule ligne (vous pouvez cependant tester chaque étape du pré-traitement de votre côté, ce qui pourrait déjà être un bon exercice en soi)._

#### **Attention, même avec la fonction, il reste toujours quelques imperfections sur lesquelles vous pourrez vous pencher.**

In [ ]:
# Chargement des tables :

customers = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_customers_dataset.csv')
items = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_items_dataset.csv')
payments = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_order_reviews_dataset.csv')
orders = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_orders_dataset.csv')
products = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_products_dataset.csv')
sellers = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/olist_sellers_dataset.csv')
names = pd.read_csv('https://raw.githubusercontent.com/dujiaying/olist/master/data/product_category_name_translation.csv')

In [ ]:
def clean_olist(customers, items, payments, reviews, orders, products, sellers, names, merge_all=True):
    """
    Fonction de nettoyage et de transformation des différentes tables opérationnelles Olist.
    Si merge_all = True, la fonction renvoie une seule dataframe prête pour l'analyse de cohorte.
    Si merge_all = False, la fonction renvoie 4 dataframes distinctes à fusionner manuellement.
    """

    # Fusion des différentes tables entre elles
    merged = items.merge(products, on='product_id', how='outer')\
                   .merge(sellers, on='seller_id', how='outer')\
                   .merge(orders, on='order_id', how='outer')\
                   .merge(payments, on='order_id', how='outer')\
                   .merge(reviews, on='order_id', how='outer')\
                   .merge(customers, on='customer_id', how='outer')

    # Suppression des colonnes non pertinentes pour l'analyse
    merged = merged.drop(columns=['payment_sequential',
                                  'customer_id',  # Identifiant client non pertinent pour l'analyse
                                  'customer_zip_code_prefix', 'customer_city',
                                  'seller_zip_code_prefix', 'seller_city',  # Données redondantes
                                  'review_id', 'review_comment_title', 'review_comment_message',
                                  'review_creation_date', 'review_answer_timestamp',  # Les commentaires ne sont pas analysés
                                  'order_approved_at', 'order_delivered_carrier_date', 'shipping_limit_date',
                                  'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                                  'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'])  # Colonnes non pertinentes

    # Calcul du délai de livraison réel par rapport à l'estimation
    merged['delivery_delay'] = pd.to_datetime(merged.order_delivered_customer_date) - pd.to_datetime(merged.order_estimated_delivery_date)
    merged['delivery_delay'] = merged['delivery_delay'].dt.round('d')  # Arrondir à des jours entiers
    merged = merged.drop(columns=['order_delivered_customer_date', 'order_estimated_delivery_date'])  # Supprimer les colonnes utilisées pour le calcul

    # Calcul de la quantité totale de chaque produit par commande
    quantity_table = merged[['order_id', 'order_item_id']].groupby('order_id').max().reset_index()
    quantity_table.columns = ['order_id', 'order_item_quantity']  # Renommer les colonnes
    merged = merged.merge(quantity_table, on='order_id', how='outer')  # Fusionner les données calculées

    # Suppression des doublons liés aux paiements
    merged = merged.drop_duplicates(subset=['order_id', 'payment_value'])

    # Calcul du nombre maximum d'installments (paiements échelonnés) pour chaque commande
    installments_table = merged[['order_id', 'payment_installments']].groupby('order_id').max().reset_index()
    installments_table.columns = ['order_id', 'payment_instalments']  # Renommer les colonnes
    merged = merged.merge(installments_table, on='order_id', how='outer')  # Fusionner les données

    # Calcul du nombre de types de paiement utilisés par commande
    types_table = merged[['order_id', 'payment_type']].groupby('order_id').nunique().reset_index()
    types_table.columns = ['order_id', 'payment_types']  # Renommer les colonnes
    merged = merged.merge(types_table, on='order_id', how='outer')

    # Somme des valeurs des paiements pour chaque commande
    payment_table = merged[['order_id', 'payment_value']].groupby('order_id').sum().reset_index()
    payment_table.columns = ['order_id', 'total_payment_value']  # Renommer les colonnes
    merged = merged.merge(payment_table, on='order_id', how='outer')

    # Suppression des doublons de commande et de produit
    merged = merged.drop_duplicates(subset=['order_id', 'product_id'])

    # Suppression des anciennes colonnes non nécessaires
    merged = merged.drop(columns=['order_item_id', 'payment_type', 'payment_installments', 'payment_value'])

    # Calcul du prix total par produit par commande (incluant la quantité)
    merged['order_item_value'] = merged.order_item_quantity * (merged.price + merged.freight_value)

    # Suppression des valeurs manquantes sur les identifiants produit
    merged = merged[~merged['product_id'].isna()].reset_index(drop=True)

    # Ajout du préfixe 'BR-' aux états pour indiquer le pays (Brésil)
    merged.customer_state = merged.customer_state.apply(lambda x: 'BR-' + x)
    merged.seller_state = merged.seller_state.apply(lambda x: 'BR-' + x)

    # Traduction des catégories de produits du portugais vers l'anglais
    portuguese = list(list(names.to_dict().values())[0].values())  # Catégories en portugais
    english = list(list(names.to_dict().values())[1].values())  # Catégories traduites en anglais
    language_pairs = {k: v for k, v in zip(portuguese, english)}
    language_pairs['nan'] = np.nan
    language_pairs['pc_gamer'] = 'pc_gamer'
    language_pairs['portateis_cozinha_e_preparadores_de_alimentos'] = 'portable_kitchen_food_preparers'
    merged['product_category_name'] = merged['product_category_name'].apply(lambda x: language_pairs[str(x)])

    # Si l'option merge_all est activée, renvoyer un dataframe unique avec les colonnes pertinentes
    if merge_all:
        return merged[['order_id', 'customer_unique_id', 'seller_id', 'product_id', 'product_category_name',
                       'price', 'freight_value', 'order_item_quantity', 'order_item_value',
                       'total_payment_value', 'payment_types', 'payment_instalments', 'review_score',
                       'order_purchase_timestamp', 'order_status', 'delivery_delay',
                       'customer_state', 'seller_state']]

    else:
        # Si merge_all est désactivé, renvoyer 4 dataframes séparés
        orders_info = merged[['order_id', 'customer_unique_id', 'seller_id', 'product_id',
                              'price', 'freight_value', 'order_item_quantity', 'order_item_value',
                              'total_payment_value', 'payment_types', 'payment_instalments', 'review_score',
                              'order_purchase_timestamp', 'order_status', 'delivery_delay']]
        customers_info = merged[['customer_unique_id', 'customer_state']]
        sellers_info = merged[['seller_id', 'seller_state']]
        product_info = merged[['product_id', 'product_category_name']]
        return (orders_info, product_info, customers_info, sellers_info)

In [ ]:
# Appel de la fonction 'clean_olist' :

merged = clean_olist(customers,
                     items,
                     payments,
                     reviews,
                     orders,
                     products,
                     sellers,
                     names)

In [ ]:
merged.head() # On visualise rapidement le dataframe

In [ ]:
# Importation de la bibliothèque 'missingno', qui est utilisée pour visualiser les données manquantes dans des ensembles de données :

import missingno as msno

msno.matrix(merged) # Application de la méthode '.matrix()' sur le dataframe.

Ce code génère une **matrice visuelle** permettant de voir la répartition et la densité des **valeurs manquantes** dans le DataFrame `merged`.

#### **C’est une manière efficace de diagnostiquer les problèmes liés aux données manquantes.**

In [ ]:
merged.info() # On affiche les informations du dataframe.

In [ ]:
merged.to_csv('clean_olist.csv', index=False) # Il ne nous reste plus qu'à exporter le jeu de données.

# Utilité de la fonction `clean_olist()`

Comme nous l'avons constaté, la fonction `clean_olist` est conçue pour **nettoyer** et **transformer** plusieurs **tables de données** liées à la plateforme Olist (clients, commandes, produits, paiements, etc.), en vue **d'analyses**.

> Elle fusionne ces tables sur des identifiants communs, supprime les colonnes non pertinentes pour l’analyse et calcule plusieurs indicateurs, comme les délais de livraison, le total des paiements ou la quantité de produits commandés. Elle peut également traduire les catégories de produits du portugais vers l’anglais. Selon l’option `merge_all`, elle renvoie soit un seul DataFrame consolidé pour l’analyse, soit quatre DataFrames distinctes pour des analyses spécifiques.
